# Notebook 01 - Image Acquisition (Input Stage)

**Objective:** Load the PlantVillage dataset from `data/raw/plant_village/`, parse folder names correctly, and create a structured inventory with proper crop/disease mapping.

**Dataset Source:** PlantVillage — `emmarex/plantdisease` on Kaggle  
**Goal:** Build a clean metadata DataFrame mapping each image to its crop and disease label.

## Import Packages

In [1]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})
sns.set_palette('viridis')

print('✅ Package imported successfully')

✅ Package imported successfully


## Define Paths

In [2]:
# ── Define all project paths ─────────────────────────────────────────────────
from pathlib import Path
import os

def get_project_root():
    """Returns correct project root in both Jupyter and Colab."""
    if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
        # Colab: runner sets CWD to notebooks/ via os.chdir()
        nb_dir = Path(os.getcwd())
    else:
        # Jupyter: use %pwd which reflects actual notebook location
        import IPython
        ip = IPython.get_ipython()
        nb_dir = Path(ip.run_line_magic('pwd', '')).resolve() if ip else Path('.').resolve()
    return nb_dir.parent  # notebooks/ -> project root

PROJECT_ROOT  = get_project_root()
DATA_DIR      = PROJECT_ROOT / 'data'
RAW_DIR       = DATA_DIR / 'raw' / 'plant_village'
METADATA_DIR  = DATA_DIR / 'metadata'
PROCESSED_DIR = DATA_DIR / 'processed'

METADATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RAW_DIR      : {RAW_DIR}')
print(f'Exists       : {RAW_DIR.exists()}')


PROJECT_ROOT : /content/drive/MyDrive/Project2_AgroLens_AI
RAW_DIR      : /content/drive/MyDrive/Project2_AgroLens_AI/data/raw/plant_village


Exists       : True


## Parse Folder Names Function Creation

**Handle all underscore patterns:**
- `Pepper_bell___Bacterial_spot` → crop='Pepper bell', disease='Bacterial spot'
- `Tomato___Early_blight` → crop='Tomato', disease='Early blight'
- `Potato__Late_blight` → crop='Potato', disease='Late blight'
- `Tomato_Tomato_YellowLeaf_Curl_Virus` → crop='Tomato', disease='Tomato YellowLeaf Curl Virus'
- `Tomato_healthy` → crop='Tomato', disease='healthy', type='healthy'


In [3]:
def parse_label(folder_name):
    """
    Extract crop and disease label from a folder name by using capitalization rules.
    Handles irregular underscores and identifies disease if any word (after crop)
    starts with an uppercase letter; otherwise labels it as healthy.

    Returns structured output with:
       - crop name
       - disease name
       - type (disease / healthy)
    """
    parts = [p for p in folder_name.split("_") if p]  # remove empty strings
    
    crop = parts[0]
    remaining = parts[1:]
    
    disease_words = []
    
    for word in remaining:
        if word[0].isupper():
            disease_words.append(word)
        elif disease_words:
            disease_words.append(word)

    # Determine final label
    if disease_words:
        return {
            "crop": crop,
            "disease": " ".join(disease_words),
            "type": "disease"
        }
    else:
        return {
            "crop": crop,
            "disease": "healthy",
            "type": "healthy"
        }
    
print('✅ Function created successfully')

✅ Function created successfully


## Testing Folder Name Parser
Validates the `parse_label()` function against different folder name formats.

### Test Cases Covered:
- Multiple underscores (`__`, `___`)
- Disease names with capitalization
- Healthy labels (no capitalized disease words)
- Mixed formatting edge cases

In [4]:
test_cases = [
    'Pepper__Bacterial_spot',
    'Tomato___Early_blight',
    'Potato___Late_blight',
    'Tomato__Target_Spot',
    'Tomato_healthy'
]

print("Testing Folder Name Parser")
print("="*100)

for test in test_cases:
    result = parse_label(test)
    print(f"{test:40} → Crop: {result['crop']:30} | Disease: {result['disease']}")

print("="*100)

Testing Folder Name Parser
Pepper__Bacterial_spot                   → Crop: Pepper                         | Disease: Bacterial spot
Tomato___Early_blight                    → Crop: Tomato                         | Disease: Early blight
Potato___Late_blight                     → Crop: Potato                         | Disease: Late blight
Tomato__Target_Spot                      → Crop: Tomato                         | Disease: Target Spot
Tomato_healthy                           → Crop: Tomato                         | Disease: healthy


In [5]:
def build_dataset_inventory(raw_dir: Path) -> pd.DataFrame:
    """
    Scan raw_dir and build DataFrame of all images with metadata.
    
    Expected structure:
        data/raw/plant_village/
            ├─ Pepper__bell___Bacterial_spot/
            │  └─ image1.jpg, image2.jpg, ...
            ├─ Tomato___Early_blight/
            │  └─ image1.jpg, image2.jpg, ...
            └─ ...
    """
    records = []
    image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

    # Validate raw_dir exists
    if not raw_dir.exists():
        print(f"❌ Error: {raw_dir} does not exist")
        return pd.DataFrame()
    
    if not raw_dir.is_dir():
        print(f"❌ Error: {raw_dir} is not a directory")
        return pd.DataFrame()

    print(f"📁 Scanning: {raw_dir}")
    
    try:
        disease_folders = sorted([d for d in raw_dir.iterdir() if d.is_dir()])
    except PermissionError:
        print(f"❌ Permission denied reading {raw_dir}")
        return pd.DataFrame()
    
    if not disease_folders:
        print(f"⚠️  No disease folders found in {raw_dir}")
        return pd.DataFrame()
    
    print(f"Found {len(disease_folders)} disease class folders")
    print()
    
    # Process each disease folder
    for disease_folder in disease_folders:
        folder_name = disease_folder.name
        meta = parse_label(folder_name)
        
        try:
            img_files = [f for f in disease_folder.iterdir() if f.suffix.lower() in image_extensions]
        except PermissionError:
            print(f"⚠️  Skipped {folder_name} (permission denied)")
            continue
        
        if not img_files:
            print(f"⚠️  No images in {folder_name}")
            continue
        
        print(f"✓ {folder_name:45} ({len(img_files):5} images) → {meta['crop']:10} | {meta['disease']}")
        
        # Add each image as a record
        for img_path in img_files:
            records.append({
                'path': str(img_path),
                'filename': img_path.name,
                'extension': img_path.suffix.lower(),
                'label': folder_name,                    # Original folder name
                'crop': meta['crop'],                    # Cleaned crop name
                'disease': meta['disease'],              # Cleaned disease name
            })
    
    if not records:
        print("\n❌ No images found!")
        return pd.DataFrame()
    
    # Create DataFrame and remove duplicates
    df = pd.DataFrame(records).drop_duplicates(subset='path')
    print(f"\n✅ Inventory created: {len(df):,} images")
    
    return df


# Build the inventory
df = build_dataset_inventory(RAW_DIR)

if not df.empty:
    print(f"\n✓ DataFrame shape: {df.shape}")
else:
    print("\n⚠️  Empty DataFrame - check that data/raw/plant_village/ exists and contains images")

📁 Scanning: /content/drive/MyDrive/Project2_AgroLens_AI/data/raw/plant_village


Found 15 disease class folders



✓ Pepper__bell___Bacterial_spot                 (  997 images) → Pepper     | Bacterial spot


✓ Pepper__bell___healthy                        ( 1478 images) → Pepper     | healthy


✓ Potato___Early_blight                         ( 1000 images) → Potato     | Early blight


✓ Potato___Late_blight                          ( 1000 images) → Potato     | Late blight
✓ Potato___healthy                              (  152 images) → Potato     | healthy
✓ Tomato_Bacterial_spot                         ( 2127 images) → Tomato     | Bacterial spot


✓ Tomato_Early_blight                           ( 1000 images) → Tomato     | Early blight
✓ Tomato_Late_blight                            ( 1909 images) → Tomato     | Late blight
✓ Tomato_Leaf_Mold                              (  952 images) → Tomato     | Leaf Mold
✓ Tomato_Septoria_leaf_spot                     ( 1771 images) → Tomato     | Septoria leaf spot


✓ Tomato_Spider_mites_Two_spotted_spider_mite   ( 1676 images) → Tomato     | Spider mites Two spotted spider mite
✓ Tomato__Target_Spot                           ( 1404 images) → Tomato     | Target Spot
✓ Tomato__Tomato_YellowLeaf__Curl_Virus         ( 3208 images) → Tomato     | Tomato YellowLeaf Curl Virus
✓ Tomato__Tomato_mosaic_virus                   (  373 images) → Tomato     | Tomato mosaic virus
✓ Tomato_healthy                                ( 1591 images) → Tomato     | healthy



✅ Inventory created: 20,638 images

✓ DataFrame shape: (20638, 6)


## Dataset Summary & Statistics

In [6]:
if not df.empty:
    print('\n' + '='*80)
    print('DATASET SUMMARY')
    print('='*80)
    
    total_images = len(df)
    total_classes = df['label'].nunique()
    unique_crops = df['crop'].nunique()
    unique_diseases = df['disease'].nunique()
    image_formats = df['extension'].unique().tolist()
    
    print(f"  Total images     : {total_images:,}")
    print(f"  Total classes    : {total_classes}")
    print(f"  Unique crops     : {unique_crops}")
    print(f"  Unique diseases  : {unique_diseases}")
    print(f"  Image formats    : {', '.join(image_formats)}")
    print()
    
    # Images per crop
    print("Images per crop:")
    crop_dist = df.groupby('crop').size().sort_values(ascending=False)
    print(crop_dist.to_string())
    print()
    
    # Images per disease
    print("\nImages per disease:")
    disease_dist = df.groupby('disease').size().sort_values(ascending=False).head(15)
    print(disease_dist.to_string())



DATASET SUMMARY
  Total images     : 20,638
  Total classes    : 15
  Unique crops     : 3
  Unique diseases  : 10
  Image formats    : .jpg, .png, .jpeg

Images per crop:
crop
Tomato    16011
Pepper     2475
Potato     2152


Images per disease:
disease
healthy                                 3221
Tomato YellowLeaf Curl Virus            3208
Bacterial spot                          3124
Late blight                             2909
Early blight                            2000
Septoria leaf spot                      1771
Spider mites Two spotted spider mite    1676
Target Spot                             1404
Leaf Mold                                952
Tomato mosaic virus                      373


## Create Output Folder Structure & Mapping File Storage

In [7]:
if not df.empty:
    # ── Save inventory to metadata folder ──────────────────────────────────────────
    inventory_path = METADATA_DIR / 'inventory.csv'
    df.to_csv(inventory_path, index=False)
    print(f"✓ Inventory saved: {inventory_path}")
    print(f"  Size: {len(df):,} rows")
    
    # ── Create crop/disease mapping file ───────────────────────────────────────────
    # This maps cleaned names back to original folder names
    mapping_records = []
    for _, row in df.iterrows():
        mapping_records.append({
            'label': row['label'],
            'crop': row['crop'],
            'disease': row['disease']
        })
    
    mapping_df = pd.DataFrame(mapping_records).drop_duplicates(subset='label')
    mapping_path = METADATA_DIR / 'crop_disease_mapping.csv'
    mapping_df.to_csv(mapping_path, index=False)
    print(f"\n✓ Mapping file saved: {mapping_path}")
    print(f"  Unique labels: {len(mapping_df)}")
    print(f"\n  Mapping Preview:")
    print(mapping_df.to_string(index=False))
    
    # ── Create folder structure tracking ──────────────────────────────────────────
    print("\n" + "="*80)
    print("OUTPUT FOLDER STRUCTURE")
    print("="*80)
    print(f"\ndata/")
    print(f"├─ raw/")
    print(f"│  └─ plant_village/          ← Input (PlantVillage dataset)")
    print(f"│     ├─ Pepper__bell___Bacterial_spot/")
    print(f"│     ├─ Tomato___Early_blight/")
    print(f"│     └─ ... ({len(mapping_df)} disease classes)")
    print(f"├─ metadata/                 ← Metadata (Created in this stage)")
    print(f"│  ├─ inventory.csv          ← All images with crop/disease labels")
    print(f"│  └─ crop_disease_mapping.csv ← Mapping of folder names to clean labels")
    print("\n" + "="*80)
    
else:
    print("❌ Cannot proceed - DataFrame is empty")

✓ Inventory saved: /content/drive/MyDrive/Project2_AgroLens_AI/data/metadata/inventory.csv
  Size: 20,638 rows



✓ Mapping file saved: /content/drive/MyDrive/Project2_AgroLens_AI/data/metadata/crop_disease_mapping.csv
  Unique labels: 15

  Mapping Preview:
                                      label   crop                              disease
              Pepper__bell___Bacterial_spot Pepper                       Bacterial spot
                     Pepper__bell___healthy Pepper                              healthy
                      Potato___Early_blight Potato                         Early blight
                       Potato___Late_blight Potato                          Late blight
                           Potato___healthy Potato                              healthy
                      Tomato_Bacterial_spot Tomato                       Bacterial spot
                        Tomato_Early_blight Tomato                         Early blight
                         Tomato_Late_blight Tomato                          Late blight
                           Tomato_Leaf_Mold Tomato            